In [33]:
from google.colab import files

uploaded = files.upload()

Saving electricity_consumption.csv to electricity_consumption (1).csv


Ingest

In [34]:
import pandas as pd

df = pd.read_csv("electricity_consumption.csv")

print("Dataset loaded successfully!")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

df.head()

Dataset loaded successfully!
Number of rows: 612
Number of columns: 3


,date,sector,consumption
0,2018-01-01,total,13156.104056
1,2018-02-01,total,12097.296631
2,2018-03-01,total,14028.979493
3,2018-04-01,total,13604.821689
4,2018-05-01,total,13934.682725


In [35]:
# Check the column names and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         612 non-null    object 
 1   sector       612 non-null    object 
 2   consumption  612 non-null    float64
dtypes: float64(1), object(2)
memory usage: 14.5+ KB


Inspect

In [36]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Check for duplicate rows
print("\nDuplicate rows:")
print(df.duplicated().sum())

Missing values:
date           0
sector         0
consumption    0
dtype: int64

Duplicate rows:
0


In [37]:
# Check the different sector categories
print(df["sector"].value_counts())

sector
total               102
local               102
local_commercial    102
local_domestic      102
exports             102
losses              102
Name: count, dtype: int64


In [38]:
# View basic statistics for the consumption column
df["consumption"].describe()

,consumption
count,612.000000
mean,7147.304797
std,5897.031842
min,25.350000
25%,1394.034742
50%,5721.178984
75%,12812.723429
max,17563.330993


In [39]:
print("Data types:")
print(df.dtypes)

print("\nSector categories:")
print(df["sector"].unique())

print("\nConsumption summary:")
print(df["consumption"].describe())

Data types:
date            object
sector          object
consumption    float64
dtype: object

Sector categories:
['total' 'local' 'local_commercial' 'local_domestic' 'exports' 'losses']

Consumption summary:
count      612.000000
mean      7147.304797
std       5897.031842
min         25.350000
25%       1394.034742
50%       5721.178984
75%      12812.723429
max      17563.330993
Name: consumption, dtype: float64


Prepare the dataset

In [40]:
# Convert date column into datetime format
df["date"] = pd.to_datetime(df["date"])

# Check the updated data types
print(df.dtypes)

date           datetime64[ns]
sector                 object
consumption           float64
dtype: object


In [41]:
print("Negative consumption values:")
print((df["consumption"] < 0).sum())

Negative consumption values:
0


In [42]:
print("Prepared dataset:")
print(df.head())

print("\nDataset shape:", df.shape)

Prepared dataset:
        date sector   consumption
0 2018-01-01  total  13156.104056
1 2018-02-01  total  12097.296631
2 2018-03-01  total  14028.979493
3 2018-04-01  total  13604.821689
4 2018-05-01  total  13934.682725

Dataset shape: (612, 3)


Validate the prepared dataset

In [43]:
# Validate the prepared dataset

print("1. Check missing values:")
print(df.isnull().sum())

print("\n2. Check duplicate rows:")
print(df.duplicated().sum())

print("\n3. Check invalid dates:")
print(df["date"].isna().sum())

print("\n4. Check negative consumption:")
print((df["consumption"] < 0).sum())

print("\n5. Check required columns:")
print(df.columns.tolist())

print("\n6. Check dataset shape:")
print(df.shape)

1. Check missing values:
date           0
sector         0
consumption    0
dtype: int64

2. Check duplicate rows:
0

3. Check invalid dates:
0

4. Check negative consumption:
0

5. Check required columns:
['date', 'sector', 'consumption']

6. Check dataset shape:
(612, 3)


Saved dataset

In [44]:
df.to_csv("electricity_consumption_cleaned.csv", index=False)

print("Prepared dataset saved successfully!")

Prepared dataset saved successfully!


In [45]:
from google.colab import files

files.download("electricity_consumption_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Data lineage manifest

In [46]:
import json
import hashlib
from datetime import datetime, timezone

# Function to calculate file fingerprint
def get_file_hash(filename):
    with open(filename, "rb") as file:
        return hashlib.sha256(file.read()).hexdigest()

# Create the manifest
manifest = {
    "pipeline_name": "MyHome Energy Calculator - Data Preparation",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source": {
        "filename": "electricity_consumption.csv",
        "source_url": "https://open.dosm.gov.my/data-catalogue/electricity_consumption",
        "sha256": get_file_hash("electricity_consumption.csv")
    },
    "output": {
        "filename": "electricity_consumption_cleaned.csv",
        "sha256": get_file_hash("electricity_consumption_cleaned.csv"),
        "rows": len(df),
        "columns": df.columns.tolist()
    },
    "preparation_steps": [
        "Converted the date column from text to datetime format.",
        "Checked for missing values.",
        "Checked for duplicate rows.",
        "Checked for invalid dates.",
        "Checked for negative consumption values."
    ],
    "validation_results": {
        "missing_values": int(df.isnull().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "invalid_dates": int(df["date"].isna().sum()),
        "negative_consumption": int((df["consumption"] < 0).sum()),
        "row_count_preserved": len(df) == 612
    }
}

# Save the manifest
with open("week3_manifest.json", "w") as file:
    json.dump(manifest, file, indent=4)

print("Data lineage manifest created successfully!")
print(json.dumps(manifest, indent=4))

Data lineage manifest created successfully!
{
    "pipeline_name": "MyHome Energy Calculator - Data Preparation",
    "created_at": "2026-11-04T15:16:27.742375+00:00",
    "source": {
        "filename": "electricity_consumption.csv",
        "source_url": "https://open.dosm.gov.my/data-catalogue/electricity_consumption",
        "sha256": "365918d0cfe987f2eac8529baf4fbffa055b1eee0c4ed595bd477f7a0b2d195f"
    },
    "output": {
        "filename": "electricity_consumption_cleaned.csv",
        "sha256": "50e27bb3a301f859f17481a2307d8367db0f1e5fd8cd39822a769ee766377b6a",
        "rows": 612,
        "columns": [
            "date",
            "sector",
            "consumption"
        ]
    },
    "preparation_steps": [
        "Converted the date column from text to datetime format.",
        "Checked for missing values.",
        "Checked for duplicate rows.",
        "Checked for invalid dates.",
        "Checked for negative consumption values."
    ],
    "validation_results": {


Pipeline Notebook

In [47]:
import pandas as pd
import os

# Step 1: Check and load source dataset
source_file = "electricity_consumption.csv"

if not os.path.exists(source_file):
    raise FileNotFoundError("Source CSV file not found!")

df = pd.read_csv(source_file)

print("Source dataset loaded:", df.shape)

# Step 2: Check required columns
required_columns = ["date", "sector", "consumption"]

if not all(col in df.columns for col in required_columns):
    raise ValueError("Required columns are missing!")

print("Required columns are available.")

# Step 3: Inspect dataset
print("\nDataset information:")
df.info()

# Step 4: Prepare dataset
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Step 5: Validate dataset
assert df.isnull().sum().sum() == 0, "Missing values found!"
assert df.duplicated().sum() == 0, "Duplicate rows found!"
assert df["date"].isna().sum() == 0, "Invalid dates found!"
assert (df["consumption"] < 0).sum() == 0, "Negative consumption found!"

print("\nAll validation checks passed!")

# Step 6: Export prepared dataset
output_file = "electricity_consumption_cleaned.csv"

df.to_csv(output_file, index=False, float_format="%.17g")

print("\nPrepared dataset exported:", output_file)
print("Final dataset shape:", df.shape)


Source dataset loaded: (612, 3)
Required columns are available.

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         612 non-null    object 
 1   sector       612 non-null    object 
 2   consumption  612 non-null    float64
dtypes: float64(1), object(2)
memory usage: 14.5+ KB

All validation checks passed!

Prepared dataset exported: electricity_consumption_cleaned.csv
Final dataset shape: (612, 3)


Repeatability test

In [48]:

import hashlib
import numpy as np

def calculate_hash(filename):
    with open(filename, "rb") as file:
        return hashlib.sha256(file.read()).hexdigest()

# First execution
first_hash = calculate_hash("electricity_consumption_cleaned.csv")

# Second execution
df_repeat = pd.read_csv("electricity_consumption.csv")
df_repeat["date"] = pd.to_datetime(df_repeat["date"])
df_repeat.to_csv(
    "electricity_consumption_cleaned_repeat.csv",
    index=False,
    float_format="%.17g"
)

second_hash = calculate_hash("electricity_consumption_cleaned_repeat.csv")

print("First execution hash:", first_hash)
print("Second execution hash:", second_hash)

print("\nSame output file:", first_hash == second_hash)

# Compare original and prepared consumption values
original = pd.read_csv("electricity_consumption.csv")
cleaned = pd.read_csv("electricity_consumption_cleaned.csv")

print(
    "Consumption values preserved:",
    np.allclose(
        original["consumption"],
        cleaned["consumption"],
        rtol=1e-12,
        atol=1e-12
    )
)


First execution hash: 7663e8045981ab53cbdae7520a29389b060c13bfe5c2d56a964d7a397052edca
Second execution hash: 7663e8045981ab53cbdae7520a29389b060c13bfe5c2d56a964d7a397052edca

Same output file: True
Consumption values preserved: True


Check original and cleaned dataset

In [49]:
# Compare original and prepared datasets

original = pd.read_csv("electricity_consumption.csv")
cleaned = pd.read_csv("electricity_consumption_cleaned.csv")

print("Original data types:")
print(original.dtypes)

print("\nPrepared data types:")
print(df.dtypes)

print("\nOriginal shape:", original.shape)
print("Prepared shape:", cleaned.shape)

print("\nAre the visible data values the same?")
print(original.equals(cleaned))

Original data types:
date            object
sector          object
consumption    float64
dtype: object

Prepared data types:
date           datetime64[ns]
sector                 object
consumption           float64
dtype: object

Original shape: (612, 3)
Prepared shape: (612, 3)

Are the visible data values the same?
False
